# Sentiment Classification


## Loading the dataset

## Train test split

In [2]:
from keras.datasets import imdb

vocab_size = 10000 #vocab size

import numpy as np
# save np.load
np_load_old = np.load

# modify the default parameters of np.load
np.load = lambda *a,**k: np_load_old(*a, allow_pickle=True, **k)

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size) # vocab_size is no.of words to consider from the dataset, ordering based on frequency.

In [3]:
# restore np.load for future normal usage
np.load = np_load_old

In [4]:
print(len(x_train), 'train sequences')
print(len(x_test), 'test sequences')

25000 train sequences
25000 test sequences


In [6]:
from keras.preprocessing.sequence import pad_sequences
from keras.preprocessing import sequence
vocab_size = 10000 #vocab size
maxlen = 300  #number of word used from each review

x_train = sequence.pad_sequences(x_train, maxlen=maxlen)
x_test = sequence.pad_sequences(x_test, maxlen=maxlen)
print('x_train shape:', x_train.shape)
print('x_test shape:', x_test.shape)

x_train shape: (25000, 300)
x_test shape: (25000, 300)


## Build Keras Embedding Layer Model
We can think of the Embedding layer as a dicionary that maps a index assigned to a word to a word vector. This layer is very flexible and can be used in a few ways:

* The embedding layer can be used at the start of a larger deep learning model. 
* Also we could load pre-train word embeddings into the embedding layer when we create our model.
* Use the embedding layer to train our own word2vec models.

The keras embedding layer doesn't require us to onehot encode our words, instead we have to give each word a unqiue intger number as an id. For the imdb dataset we've loaded this has already been done, but if this wasn't the case we could use sklearn [LabelEncoder](http://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html).

In [8]:
from keras.models import Sequential
from keras.layers import Dense, Embedding
from keras.layers import LSTM
max_features = 10000

model = Sequential()
model.add(Embedding(max_features, 128))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation='sigmoid'))

Instructions for updating:
Colocations handled automatically by placer.
Instructions for updating:
Please use `rate` instead of `keep_prob`. Rate should be set to `rate = 1 - keep_prob`.


In [9]:
# try using different optimizers and different optimizer configs
model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

In [11]:
batch_size = 32

model.fit(x_train, y_train,
          batch_size=batch_size,
          epochs=15,
          validation_data=(x_test, y_test))
score, acc = model.evaluate(x_test, y_test,
                            batch_size=batch_size)
print('Test score:', score)
print('Test accuracy:', acc)

Instructions for updating:
Use tf.cast instead.
Train on 25000 samples, validate on 25000 samples
Epoch 1/15
25000/25000 [==============================] - 425s 17ms/step - loss: 0.4643 - acc: 0.7795 - val_loss: 0.4321 - val_acc: 0.8077
Epoch 2/15
25000/25000 [==============================] - 417s 17ms/step - loss: 0.3304 - acc: 0.8669 - val_loss: 0.3617 - val_acc: 0.8494
Epoch 3/15
25000/25000 [==============================] - 416s 17ms/step - loss: 0.2796 - acc: 0.8871 - val_loss: 0.3778 - val_acc: 0.8584
Epoch 4/15
25000/25000 [==============================] - 405s 16ms/step - loss: 0.2906 - acc: 0.8786 - val_loss: 0.3326 - val_acc: 0.8595
Epoch 5/15
25000/25000 [==============================] - 447s 18ms/step - loss: 0.1845 - acc: 0.9289 - val_loss: 0.3689 - val_acc: 0.8574
Epoch 6/15
25000/25000 [==============================] - 441s 18ms/step - loss: 0.1355 - acc: 0.9500 - val_loss: 0.3901 - val_acc: 0.8683
Epoch 7/15
25000/25000 [==============================] - 424s 17ms/

## Retrive the output of each layer in keras for a given single test sample from the trained model you built

In [17]:
import pandas as pd

imdb_data=pd.read_csv('imdb_master.csv', encoding = "ISO-8859-1")
print(imdb_data.shape)

(100000, 5)


In [27]:
# save model and architecture to single file
model.save("model.h5")
print("Saved model to disk")

Saved model to disk


In [28]:
from keras.models import load_model
# load model
model = load_model('model.h5')
# summarize model.
model.summary()

_________________________________________________________________
Layer (type)                 Output Shape              Param #   
embedding_1 (Embedding)      (None, None, 128)         1280000   
_________________________________________________________________
lstm_1 (LSTM)                (None, 128)               131584    
_________________________________________________________________
dense_1 (Dense)              (None, 1)                 129       
Total params: 1,411,713
Trainable params: 1,411,713
Non-trainable params: 0
_________________________________________________________________


In [30]:
# evaluate the model
score = model.evaluate(x_test, y_test, verbose=0)
print("%s: %.2f%%" % (model.metrics_names[1], score[1]*100))

acc: 86.02%


In [31]:
imdb_data.head()

,Unnamed: 0,type,review,label,file
0,0,test,Once again Mr. Costner has dragged out a movie...,neg,0_2.txt
1,1,test,This is an example of why the majority of acti...,neg,10000_4.txt
2,2,test,"First of all I hate those moronic rappers, who...",neg,10001_1.txt
3,3,test,Not even the Beatles could write songs everyon...,neg,10002_3.txt
4,4,test,Brass pictures (movies is not a fitting word f...,neg,10003_3.txt


In [40]:
from tensorflow.python.keras.preprocessing.text import Tokenizer

import pandas as pd
from keras.models import Model
#import nltk
#df['tokenized_sents'] = df.apply(lambda row: nltk.word_tokenize(row['sentences']), axis=1)
tokenizer_obj = Tokenizer()
total_reviews = imdb_data['review']
tokenizer_obj.fit_on_texts(total_reviews)

test_sample_1 = "Harriet gets the story right, and will surely have its fans. However, with so much talent involved, you'll wish there was more to write home about"
test_sample_2 = "Excellent movie, must watch"
test_samples = [test_sample_1,test_sample_2]

test_samples_tokens = tokenizer_obj.texts_to_sequences(test_samples)
test_samples_tokens_pad = pad_sequences(test_samples_tokens, maxlen=maxlen)

# Intermediate model has the same input as basic model,
# but has all the intermediate layers as outputs

intermediate_model = Model(inputs=model.layers[0].input, 
                              outputs=[l.output for l in model.layers[1:]])
intermediate_model.predict(test_samples_tokens_pad) # outputs a list of 2 arrays

[array([[-0.04120603, -0.00489829, -0.00768119,  0.02204878,  0.04884697,
          0.10453726,  0.00905261,  0.01014332, -0.04550809, -0.064666  ,
          0.04509193, -0.06182883, -0.05618683,  0.13351731, -0.3302039 ,
         -0.11990441, -0.2125619 , -0.06484661, -0.09178289, -0.07237767,
          0.07481857,  0.05760352, -0.00478118,  0.09744363,  0.09023235,
         -0.15159921,  0.01148095,  0.08234882,  0.03474389, -0.00703972,
          0.3326098 ,  0.11611019,  0.0190182 , -0.02007438,  0.00976459,
         -0.06281041, -0.02102903,  0.07355807, -0.04597036, -0.21291704,
         -0.01499853, -0.22399388, -0.1084928 ,  0.22090992,  0.30443215,
         -0.07874951, -0.15904175,  0.08852388, -0.20990393,  0.0319846 ,
         -0.01111655, -0.00792384,  0.07602385,  0.02289673,  0.00460622,
         -0.24491055, -0.13629326, -0.17597371, -0.10688069, -0.19153042,
         -0.27992237,  0.00405408,  0.02969017, -0.03513441, -0.28939182,
         -0.16563775, -0.01132702,  0.

In [41]:
model.predict(test_samples_tokens_pad)

array([[0.20044385],
       [0.9620462 ]], dtype=float32)